# 1주차 · 거래량 시계열 예측 (ARIMA) 📈

DAT 학술트랙 첫 실습에 오신 걸 환영해요! 🎉 이번 주 미니 대회는 **애플(AAPL) 주식의 거래량(Volume)을 예측**하는 문제예요. 여러분이 만든 예측을 `submission.csv`로 제출하면, 웹사이트가 자동으로 점수를 매겨 리더보드에 올려줍니다.

왜 가격이 아니라 거래량이냐면 — 거래량은 **어제 값과 오늘 값이 강하게 이어지는 성질(지속성)과 완만한 추세**가 있어서, 오늘 배울 **시계열 모델(ARIMA)**이 실력을 발휘하기 좋기 때문이에요. (요일 같은 뚜렷한 주기 패턴은 이 데이터에선 거의 안 보여요 — 아래에서 직접 확인해봅니다.)

**진행 방법 (꼭 읽어주세요!)**
1. 코랩에서 셀을 **위에서부터 하나씩** 클릭하고 `Shift+Enter`로 실행하세요. 순서를 건너뛰면 에러가 나요.
2. 대부분의 셀은 *그냥 실행*하면 되고, **✏️ 표시된 셀만** 바꿔보면 됩니다.
3. 마지막 셀을 실행하면 `submission.csv`가 다운로드돼요 → 웹의 **Submit 탭**에 올리면 끝!

파이썬이 처음이어도 괜찮아요. 코드마다 한글 주석을 달아뒀으니, 천천히 읽으면서 따라오세요. 😊

- 평가 지표: **RMSE** (예측이 실제와 얼마나 다른지의 평균 — **낮을수록 좋음**)
- 기본 코드 그대로 제출해도 베이스라인(나이브 예측)은 이깁니다. 일단 끝까지 한 번 완주해보세요!

## 1. 데이터 불러오기  *(그냥 실행하세요)*

인터넷에 올려둔 CSV 파일 두 개를 표(데이터프레임)로 읽어옵니다.
- `train`: 과거 날짜별 거래량이 들어있는 **학습용** 데이터 (모델이 여기서 패턴을 배워요)
- `test`: 우리가 **예측해야 할 미래 날짜들** (정답인 거래량 컬럼은 없어요!)

실행하면 아래에 `train: (행수, 열수)` 와 표의 첫 5줄이 보이면 성공이에요.

In [ ]:
# pandas: 표(엑셀 같은) 데이터를 다루는 파이썬의 대표 라이브러리
import pandas as pd

# 데이터 파일이 올라가 있는 공개 링크 (운영진이 채워둔 것 — 수정하지 마세요!)
TRAIN_URL = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/finance/train.csv"
TEST_URL  = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/finance/test.csv"

if TRAIN_URL:
    # URL에서 CSV를 읽어 표로 만듭니다. parse_dates: date 컬럼을 '날짜 타입'으로 인식시켜요.
    train = pd.read_csv(TRAIN_URL, parse_dates=["date"])
    test  = pd.read_csv(TEST_URL,  parse_dates=["date"])
else:
    # (예비용) URL이 비어있으면 파일을 직접 업로드하는 코드예요. 보통은 실행되지 않아요.
    from google.colab import files
    up = files.upload()
    train = pd.read_csv("train.csv", parse_dates=["date"])
    test  = pd.read_csv("test.csv",  parse_dates=["date"])

# 데이터 크기 확인: (행 개수, 열 개수)
print("train:", train.shape, "| test:", test.shape)
# 표의 첫 5줄을 미리보기 (노트북은 셀의 마지막 줄을 자동으로 화면에 보여줘요)
train.head()

## 2. 데이터 살펴보기  *(그냥 실행하세요)*

모델을 만들기 전에 데이터를 **눈으로 먼저** 보는 습관! 그래프 2개를 그립니다.
- 왼쪽: 시간에 따른 거래량 흐름 → 오르락내리락하는 큰 흐름이 보여요
- 오른쪽: 요일별 평균 거래량 (0=월요일 … 4=금요일) → 막대 높이가 요일마다 크게 다르다면 **주기 패턴(계절성)**을 의심할 수 있어요. 다만 이 데이터는 **요일별 평균이 거의 비슷해서**(막대들이 고만고만해요) 요일 계절성은 뚜렷하지 않습니다. ARIMA가 실제로 활용하는 신호는 요일이 아니라 **어제 값과의 강한 연결(지속성)과 추세**예요.

In [ ]:
# matplotlib: 그래프를 그리는 라이브러리
import matplotlib.pyplot as plt

# 그래프 2개를 나란히 놓을 공간을 만듭니다 (1행 2열)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

# 왼쪽 그래프: 날짜(x축)에 따른 거래량(y축) 선 그래프
ax[0].plot(train["date"], train["volume"])
ax[0].set_title("거래량 추이")

# 오른쪽 그래프: 요일(0=월 ~ 4=금)별로 묶어 평균 거래량을 막대로
train.groupby(train["date"].dt.dayofweek)["volume"].mean().plot.bar(ax=ax[1])
ax[1].set_title("요일별 평균 거래량 (0=월)")

plt.show()  # 그래프를 화면에 표시

## 3. ✏️ 여기를 바꿔보세요 — 모델 학습

이제 핵심! **ARIMA**라는 시계열 모델을 학습시킵니다. ARIMA는 "과거 값들의 흐름을 보고 다음 값을 추측"하는 고전적이지만 강력한 모델이에요. 이 데이터는 **어제 거래량과 오늘 거래량이 강하게 이어지는 성질(지속성)**이 뚜렷해서, 그 흐름만 잘 따라가도 나이브 베이스라인을 이깁니다.

아래 코드는 **그대로 실행해도 동작하는 기본 예시**이고, 나이브 베이스라인(마지막 값 그대로 찍기)은 이깁니다. 실행하면 모델 요약표가 출력돼요 (숫자가 많아도 겁먹지 마세요, 안 읽어도 됩니다 😄).

**점수(RMSE)를 낮추는 실험 아이디어** — 한 번에 하나씩 바꾸고 제출해서 비교해보세요:
- `order=(p, d, q)` 숫자 바꾸기 → 예: `(1, 1, 1)`, `(3, 0, 3)`, `(5, 1, 2)` 등. p=과거 값 몇 개를 볼지, d=차분(추세 제거) 횟수, q=과거 오차 몇 개를 볼지.
- **주간 계절성** 실험: `ARIMA` 대신 `from statsmodels.tsa.statespace.sarimax import SARIMAX` 를 쓰고 `SARIMAX(y, order=(2,0,2), seasonal_order=(1,0,1,5))` 처럼 `5`(주 5거래일) 주기를 넣어볼 수 있어요. 다만 이 데이터는 요일 계절성이 약해서 큰 효과가 없을 수도 있으니, **직접 실험해 점수로 확인**하고 과신하진 마세요.
- 로그 변환은 이미 적용돼 있어요 (아래 주석 참고).

In [ ]:
import numpy as np  # 수치 계산 라이브러리
from statsmodels.tsa.arima.model import ARIMA  # 시계열 예측 모델

# [로그 변환] 거래량은 수천만 단위로 값이 크고 들쭉날쭉해서,
# log를 씌워 숫자를 작고 안정적으로 만든 뒤 예측합니다. (마지막에 다시 되돌려요)
y = np.log(train["volume"].astype(float).values)

# ✏️ 여기를 바꿔보세요! order=(p, d, q) 값을 바꿔가며 실험 (지금 값도 베이스라인은 이깁니다)
#    예시: (1,1,1) / (3,0,3) / (5,1,2) ... 바꿀 때마다 제출해서 RMSE를 비교해보세요.
model = ARIMA(y, order=(2, 0, 2))

# 모델 학습: 과거 데이터에서 패턴을 찾아냅니다 (몇 초 걸릴 수 있어요)
fitted = model.fit()

# 학습 결과 요약표 출력 (참고용 — 다 이해하지 않아도 괜찮아요!)
print(fitted.summary().tables[0])

## 4. test 구간 예측  *(그냥 실행하세요)*

학습한 모델로 test에 있는 **미래 날짜들의 거래량**을 예측합니다. 아까 로그를 씌웠으니, `np.exp`로 다시 원래 크기(수천만 단위)로 되돌려요. 실행하면 예측값이 붙은 표 5줄이 보입니다.

In [ ]:
# forecast(steps=N): 학습 데이터 바로 다음 시점부터 N개를 예측
# np.exp(...): 로그 변환을 원래 스케일(진짜 거래량 크기)로 되돌리기
pred = np.exp(fitted.forecast(steps=len(test)))

# 예측값을 test 표에 'prediction'이라는 새 컬럼으로 붙입니다
test["prediction"] = pred

# 예측 결과 미리보기 (거래량이니까 수백만~수천만 정도 숫자면 정상이에요)
test[["id", "date", "prediction"]].head()

## 5. 제출 파일 저장  *(실행하면 자동 다운로드)*

`id`와 `prediction` 두 컬럼만 뽑아 `submission.csv`로 저장하고, 코랩이 자동으로 **내 컴퓨터에 다운로드**해줍니다 (브라우저 하단에 파일이 떠요).

👉 다운로드된 `submission.csv`를 웹사이트의 **Submit 탭**에 업로드하면 몇 초 안에 점수가 나오고 리더보드에 반영됩니다. 제출은 여러 번 할 수 있으니 부담 없이 실험하세요!

In [ ]:
# 제출 규칙: id, prediction 두 컬럼만 있으면 됩니다
submission = test[["id", "prediction"]]
# index=False: 불필요한 행 번호는 파일에 넣지 않기
submission.to_csv("submission.csv", index=False)
print("저장 완료:", submission.shape)

try:
    # 코랩에서 실행 중이면 파일을 자동으로 다운로드해줍니다
    from google.colab import files
    files.download("submission.csv")
except Exception:
    print("코랩이 아니면 왼쪽 파일탭에서 submission.csv를 내려받으세요.")

## 🆘 막히면 여기를 보세요

| 증상 | 해결 |
|---|---|
| `NameError: name 'train' is not defined` | 위쪽 셀을 건너뛰었어요. 맨 위부터 순서대로 다시 실행하세요. |
| 데이터 로드 셀에서 URL 에러 | 인터넷 연결을 확인하고 셀을 한 번 더 실행해보세요. |
| 그래프에 한글이 □□로 깨져 보임 | 무시해도 됩니다 — 점수와는 전혀 상관없어요. |
| ARIMA 학습 시 경고(Warning)가 잔뜩 나옴 | 빨간 에러가 아닌 경고는 정상입니다. 그냥 진행하세요. |
| 코랩이 멈춘 것 같음 | 메뉴 `런타임 > 세션 다시 시작` 후 맨 위부터 재실행. |

수고했어요! 궁금한 건 언제든 운영진에게 물어보세요. 🙌